# Claim Attention Hybrid Score Validation Notebook

**Module:** IRIS Claim Attention Hybrid V1 Candidate  
**Objectif:** verifier le score hybride final candidat avec un audit detaille, des exemples metier et des controles de coherence.

Ce notebook valide que le score hybride :

- utilise des regles configurables ;
- applique une ponderation par regle et par famille ;
- respecte les plafonds de score ;
- integre le Scenario A post-inspection seulement ;
- garde le Scenario B en readiness-only ;
- ne modifie pas VHS ;
- ne modifie pas Claim Attention Score V1 ;
- reste une aide a la priorisation humaine, sans conclusion automatique.

> Vocabulaire IRIS : signal d'attention, verification, priorisation, aide a l'analyse.

## 1. Positionnement metier

Le score hybride n'est pas une preuve et ne doit pas etre lu comme une decision automatique.
Il combine des signaux explicables pour aider un gestionnaire a prioriser les dossiers a verifier.

**Statut important :** ce notebook valide une version candidate, pas une version production officielle.
Le score V1 officiel reste disponible separement. Dans Power BI, Angular ou toute restitution metier, il faut toujours filtrer explicitement par score_version afin de ne pas melanger IRIS_CLAIM_ATTENTION_V1_CANDIDATE et IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE.

Terme metier recommande : **score de priorisation enrichi par signaux metier**. Le terme hybrid reste acceptable cote code et audit technique.

Architecture logique :

```text
Feature mart V1
        +
Business rule signals configurables
        +
Post-inspection Scenario A
        ->
Claim Attention Hybrid Score V1 Candidate
```

Le Scenario B inspection -> avenant reste PARTIAL/readiness-only tant que la nature exacte du changement de garantie ou de couverture n'est pas fiablement disponible.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

BASE_DIR = Path.cwd()
if BASE_DIR.name != "IRIS_AUTO_FRAUD":
    if Path.cwd().name == "validation_scoring":
        BASE_DIR = Path.cwd().parents[1]
    elif Path.cwd().name == "notebooks":
        BASE_DIR = Path.cwd().parent

sys.path.insert(0, str(BASE_DIR))
sys.path.insert(0, str(BASE_DIR / "etl" / "dwh"))

import dwh_utils

SCORE_VERSION = "IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE"
BUSINESS_RULE_SIGNAL_VERSION = "IRIS_CLAIM_BUSINESS_RULE_SIGNAL_V1_CANDIDATE"
POST_INSPECTION_SIGNAL_VERSION = "IRIS_POST_INSPECTION_SIGNAL_V1_CANDIDATE"
FEATURE_VERSION = "IRIS_CLAIM_ATTENTION_FEATURES_V1_CANDIDATE"

CONFIG_PATH = BASE_DIR / "config" / "scoring" / "claim_attention_hybrid_v1_candidate.json"
REPORT_DIR = BASE_DIR / "data" / "quality_reports" / "scoring" / "claim_attention_hybrid_v1" / "notebook_audit"
EXPORT_REPORTS = True

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

CONFIG_PATH

WindowsPath('c:/Users/wiem/Downloads/Projet PFE/IRIS_AUTO_FRAUD/config/scoring/claim_attention_hybrid_v1_candidate.json')

## 2. Chargement de la configuration

La configuration est un point central de gouvernance : les poids et plafonds doivent etre visibles et modifiables sans changer le code.

La validation du JSON est obligatoire avant toute interpretation du score, car il controle les poids de regles, les plafonds par famille, le score maximum et les points post-inspection.


In [2]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    score_config = json.load(handle)

assert score_config["score_version"] == SCORE_VERSION
assert score_config["business_rules"]["enabled"] is True
assert score_config["post_inspection"]["enabled"] is True
assert score_config["post_inspection"]["scenario_code"] == "A_INSPECTION_TO_CLAIM"

score_config

{'score_version': 'IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE',
 'profile_name': 'CLAIM_ATTENTION_HYBRID_V1_CANDIDATE',
 'max_score': 100,
 'business_rules': {'enabled': True,
  'max_points': 70,
  'exclude_data_quality_signals': True,
  'default_rule_weight': 1.0,
  'family_weights': {'Recurrence client': 1.0,
   'Montant atypique': 1.0,
   'Chronologie': 1.0,
   'Recurrence vehicule': 1,
   'Recurrence conducteur': 1,
   'Recurrence tiers': 1,
   'Repetition garantie': 1},
  'family_caps': {'Recurrence client': 25,
   'Montant atypique': 25,
   'Chronologie': 20,
   'Recurrence vehicule': 15,
   'Recurrence conducteur': 12,
   'Recurrence tiers': 12,
   'Repetition garantie': 10},
  'rule_weights': {'CLIENT_CLAIMS_12M_HIGH': 1.0,
   'CLIENT_CLAIMS_12M_MEDIUM': 1.0,
   'CLIENT_CLAIMS_12M_LOW': 1.0,
   'CLIENT_RECENT_PREVIOUS_CLAIM': 1.0,
   'AMOUNT_HIGH_BY_GUARANTEE': 1.0,
   'AMOUNT_MEDIUM_BY_GUARANTEE': 1.0,
   'AMOUNT_LOW_BY_GUARANTEE': 1.0,
   'CLAIM_BEFORE_CONTRACT_START': 1.0,
   

In [3]:
family_caps = pd.Series(score_config["business_rules"]["family_caps"], name="family_cap_points").reset_index().rename(columns={"index": "rule_family"})
family_weights = pd.Series(score_config["business_rules"]["family_weights"], name="family_weight").reset_index().rename(columns={"index": "rule_family"})
rule_weights = pd.Series(score_config["business_rules"]["rule_weights"], name="rule_weight").reset_index().rename(columns={"index": "rule_code"})
post_points = pd.Series(score_config["post_inspection"]["confidence_points"], name="post_inspection_points").reset_index().rename(columns={"index": "confidence_level"})

family_config = family_caps.merge(family_weights, on="rule_family", how="outer")
display(family_config)
display(rule_weights)
display(post_points)

,rule_family,family_cap_points,family_weight
0,Chronologie,20,1.0
1,Montant atypique,25,1.0
2,Recurrence client,25,1.0
3,Recurrence conducteur,12,1.0
4,Recurrence tiers,12,1.0
5,Recurrence vehicule,15,1.0
6,Repetition garantie,10,1.0


,rule_code,rule_weight
0,CLIENT_CLAIMS_12M_HIGH,1.0
1,CLIENT_CLAIMS_12M_MEDIUM,1.0
2,CLIENT_CLAIMS_12M_LOW,1.0
3,CLIENT_RECENT_PREVIOUS_CLAIM,1.0
4,AMOUNT_HIGH_BY_GUARANTEE,1.0
5,AMOUNT_MEDIUM_BY_GUARANTEE,1.0
6,AMOUNT_LOW_BY_GUARANTEE,1.0
7,CLAIM_BEFORE_CONTRACT_START,1.0
8,CLAIM_SOON_AFTER_CONTRACT_START,1.0
9,CLAIM_WITHIN_90D_CONTRACT_START,1.0


,confidence_level,post_inspection_points
0,HIGH,25
1,MEDIUM,15
2,LOW,0


## 3. Connexion read-only et detection des derniers runs

Le notebook lit PostgreSQL, mais ne cree pas et ne modifie aucune table.

In [4]:
logger = dwh_utils.setup_logging("NOTEBOOK_CLAIM_ATTENTION_HYBRID_VALIDATION", log_name="notebook_claim_attention_hybrid_validation")
engine = dwh_utils.build_engine(logger)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params or {})

def latest_run(table_name: str, run_col: str, version_col: str, version: str) -> str | None:
    query = f"""
        SELECT {run_col} AS run_id
        FROM {table_name}
        WHERE {version_col} = :version
        GROUP BY {run_col}
        ORDER BY MAX(created_at) DESC
        LIMIT 1
    """
    df = read_sql(query, {"version": version})
    return None if df.empty else str(df.loc[0, "run_id"])

score_run_id = latest_run("mart.fact_claim_attention_score", "score_run_id", "score_version", SCORE_VERSION)
feature_run_id = latest_run("mart.fact_claim_scoring_features", "feature_run_id", "scoring_feature_version", FEATURE_VERSION)
business_rule_run_id = latest_run("mart.fact_claim_business_rule_signal", "signal_run_id", "signal_version", BUSINESS_RULE_SIGNAL_VERSION)
post_inspection_run_id = latest_run("mart.fact_post_inspection_attention_signal", "signal_run_id", "signal_version", POST_INSPECTION_SIGNAL_VERSION)

run_context = pd.DataFrame([{
    "score_run_id": score_run_id,
    "feature_run_id": feature_run_id,
    "business_rule_signal_run_id": business_rule_run_id,
    "post_inspection_signal_run_id": post_inspection_run_id,
}])
run_context

2026-07-08 21:17:18 | INFO     | Logger initialise | run_id=NOTEBOOK_CLAIM_ATTENTION_HYBRID_VALIDATION | log=C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\logs\notebook_claim_attention_hybrid_validation_NOTEBOOK_CLAIM_ATTENTION_HYBRID_VALIDATION.log
2026-07-08 21:17:18 | INFO     | Engine PostgreSQL : localhost:5432/iris_auto_fraud


,score_run_id,feature_run_id,business_rule_signal_run_id,post_inspection_signal_run_id
0,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE_20260...,IRIS_CLAIM_ATTENTION_FEATURES_V1_CANDIDATE_202...,IRIS_CLAIM_BUSINESS_RULE_SIGNAL_V1_CANDIDATE_2...,IRIS_POST_INSPECTION_SIGNAL_V1_CANDIDATE_20260...


## 4. Chargement des donnees d'audit

On charge uniquement les colonnes necessaires pour verifier le score hybride et produire des exemples explicables.

In [5]:
assert score_run_id is not None, "No hybrid score run found. Run compute_claim_attention_hybrid_score_v1_candidate.py first."

scores = read_sql("""
    SELECT claim_sk, claim_business_id, score_version, score_run_id, feature_run_id,
           attention_score, attention_level, confidence_level,
           main_reason_1, main_reason_2, main_reason_3, created_at
    FROM mart.fact_claim_attention_score
    WHERE score_version = :score_version
      AND score_run_id = :score_run_id
""", {"score_version": SCORE_VERSION, "score_run_id": score_run_id})

details = read_sql("""
    SELECT claim_sk, claim_business_id, score_run_id, score_version,
           signal_family, signal_code, signal_label, signal_value,
           points, severity, business_explanation, created_at
    FROM mart.fact_claim_attention_signal_detail
    WHERE score_version = :score_version
      AND score_run_id = :score_run_id
""", {"score_version": SCORE_VERSION, "score_run_id": score_run_id})

source_counts = read_sql("""
    SELECT
        (SELECT COUNT(*) FROM dwh.fact_sinistre) AS fact_sinistre_rows,
        (SELECT COUNT(*) FROM mart.fact_claim_scoring_features WHERE scoring_feature_version = :feature_version AND feature_run_id = :feature_run_id) AS feature_rows,
        (SELECT COUNT(*) FROM mart.fact_claim_business_rule_signal WHERE signal_version = :business_version AND signal_run_id = :business_run_id) AS business_rule_signal_rows,
        (SELECT COUNT(*) FROM mart.fact_post_inspection_attention_signal WHERE signal_version = :post_version AND signal_run_id = :post_run_id) AS post_inspection_signal_rows
""", {
    "feature_version": FEATURE_VERSION,
    "feature_run_id": feature_run_id,
    "business_version": BUSINESS_RULE_SIGNAL_VERSION,
    "business_run_id": business_rule_run_id,
    "post_version": POST_INSPECTION_SIGNAL_VERSION,
    "post_run_id": post_inspection_run_id,
})

print("scores", len(scores), "details", len(details))
source_counts

scores 381893 details 646781


,fact_sinistre_rows,feature_rows,business_rule_signal_rows,post_inspection_signal_rows
0,381893,381893,666441,148


## 5. Controles d'acceptation

Ces controles verifient le grain, les bornes, les niveaux, la coherence entre detail et score, et l'absence de vocabulaire accusatoire.

In [6]:
BLOCKLIST = ["fraud detected", "fraudulent", "proof of fraud", "fraude detectee", "fraude confirmee", "client fraudeur", "fraudeur"]

def contains_blocked_wording(value) -> bool:
    text_value = str(value or "").lower()
    return any(term in text_value for term in BLOCKLIST)

positive_points_by_claim = (
    details.loc[pd.to_numeric(details["points"], errors="coerce") > 0]
    .groupby("claim_sk")["points"].sum()
    if not details.empty else pd.Series(dtype="int64")
)
score_vs_detail = scores.set_index("claim_sk")["attention_score"].subtract(positive_points_by_claim, fill_value=0)

validation_checks = pd.DataFrame([{
    "score_rows": len(scores),
    "expected_fact_sinistre_rows": int(source_counts.loc[0, "fact_sinistre_rows"]),
    "score_rows_equal_fact_sinistre": len(scores) == int(source_counts.loc[0, "fact_sinistre_rows"]),
    "duplicate_score_rows": int(scores.duplicated(["claim_sk", "score_version", "score_run_id"]).sum()),
    "score_out_of_range_rows": int((~scores["attention_score"].between(0, 100)).sum()),
    "null_attention_level_rows": int(scores["attention_level"].isna().sum()),
    "null_confidence_level_rows": int(scores["confidence_level"].isna().sum()),
    "detail_point_mismatch_rows": int(score_vs_detail.ne(0).sum()),
    "null_detail_explanation_rows": int(details["business_explanation"].isna().sum() + details["business_explanation"].astype(str).str.strip().eq("").sum()),
    "accusatory_wording_rows": int(pd.concat([
        details["business_explanation"].fillna(""), scores["main_reason_1"].fillna(""),
        scores["main_reason_2"].fillna(""), scores["main_reason_3"].fillna("")
    ], ignore_index=True).map(contains_blocked_wording).sum()),
    "scenario_b_detail_rows": int(details["signal_code"].astype(str).str.contains("AVENANT|SCENARIO_B|ENDORSEMENT", case=False, regex=True).sum()),
}])

validation_checks.T

,0
score_rows,381893
expected_fact_sinistre_rows,381893
score_rows_equal_fact_sinistre,True
duplicate_score_rows,0
score_out_of_range_rows,0
null_attention_level_rows,0
null_confidence_level_rows,0
detail_point_mismatch_rows,0
null_detail_explanation_rows,0
accusatory_wording_rows,0


In [7]:
assert validation_checks.loc[0, "score_rows_equal_fact_sinistre"]
assert validation_checks.loc[0, "duplicate_score_rows"] == 0
assert validation_checks.loc[0, "score_out_of_range_rows"] == 0
assert validation_checks.loc[0, "null_attention_level_rows"] == 0
assert validation_checks.loc[0, "null_confidence_level_rows"] == 0
assert validation_checks.loc[0, "detail_point_mismatch_rows"] == 0
assert validation_checks.loc[0, "accusatory_wording_rows"] == 0
assert validation_checks.loc[0, "scenario_b_detail_rows"] == 0

"Core validation checks passed"

'Core validation checks passed'

## 6. Distribution du score

Ces sorties permettent de verifier si le score est exploitable pour une priorisation : concentration, niveaux d'attention, confidence level et score moyen par niveau.

In [8]:
score_distribution = scores["attention_level"].value_counts(dropna=False).rename_axis("attention_level").reset_index(name="rows")
score_distribution["pct"] = (score_distribution["rows"] / len(scores) * 100).round(2)
score_stats = scores["attention_score"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("attention_score")
confidence_distribution = scores["confidence_level"].value_counts(dropna=False).rename_axis("confidence_level").reset_index(name="rows")
confidence_distribution["pct"] = (confidence_distribution["rows"] / len(scores) * 100).round(2)
score_by_confidence = scores.groupby("confidence_level", dropna=False)["attention_score"].agg(["count", "mean", "median", "max"]).reset_index()

display(score_distribution)
display(score_stats)
display(confidence_distribution)
display(score_by_confidence)

,attention_level,rows,pct
0,Analyse standard,287129,75.19
1,Points a verifier,78507,20.56
2,Examen renforce suggere,16255,4.26
3,Examen prioritaire suggere,2,0.00


,attention_score
count,381893.000000
mean,15.518496
std,15.503271
min,0.000000
50%,10.000000
75%,24.000000
90%,40.000000
95%,48.000000
99%,61.000000
max,92.000000


,confidence_level,rows,pct
0,HIGH,301773,79.02
1,MEDIUM,74526,19.51
2,LOW,5594,1.46


,confidence_level,count,mean,median,max
0,HIGH,301773,15.964639,11.0,92
1,LOW,5594,30.450125,35.0,70
2,MEDIUM,74526,12.591176,8.0,70


## 7. Contribution des familles de signaux

Le score hybride doit rester explicable : chaque point du score doit venir d'une famille et d'une regle identifiee.

In [9]:
family_summary = (
    details.groupby("signal_family", dropna=False)
    .agg(signal_rows=("signal_code", "size"), distinct_claims=("claim_sk", "nunique"), total_points=("points", "sum"), mean_points=("points", "mean"), max_points=("points", "max"))
    .reset_index()
    .sort_values("total_points", ascending=False)
)
rule_summary = (
    details.groupby(["signal_family", "signal_code", "signal_label"], dropna=False)
    .agg(signal_rows=("claim_sk", "size"), distinct_claims=("claim_sk", "nunique"), total_points=("points", "sum"), mean_points=("points", "mean"))
    .reset_index()
    .sort_values(["total_points", "signal_rows"], ascending=False)
)

display(family_summary)
display(rule_summary.head(30))

,signal_family,signal_rows,distinct_claims,total_points,mean_points,max_points
3,Regles metier - Recurrence client,145238,118966,1771640,12.198185,20
1,Regles metier - Chronologie,216623,197610,1573264,7.262682,15
6,Regles metier - Recurrence vehicule,70879,66587,801112,11.302530,15
2,Regles metier - Montant atypique,64629,64629,768662,11.893453,20
4,Regles metier - Recurrence conducteur,111596,61385,711816,6.378508,10
7,Regles metier - Repetition garantie,33241,31356,266070,8.004272,10
5,Regles metier - Recurrence tiers,4515,3009,32866,7.279291,10
0,Post-inspection,60,60,975,16.250000,25


,signal_family,signal_code,signal_label,signal_rows,distinct_claims,total_points,mean_points
12,Regles metier - Recurrence client,CLIENT_CLAIMS_12M_HIGH,Recurrence client elevee sur 12 mois,50180,50180,1003600,20.000000
16,Regles metier - Recurrence conducteur,DRIVER_CLAIMS_12M_HIGH,Recurrence conducteur sur 12 mois,60940,60940,609200,9.996718
7,Regles metier - Chronologie,LONG_DECLARATION_DELAY_HIGH,Delai de declaration long,68977,68977,539457,7.820824
9,Regles metier - Montant atypique,AMOUNT_HIGH_BY_GUARANTEE,Montant eleve dans la garantie,23743,23743,474860,20.000000
14,Regles metier - Recurrence client,CLIENT_CLAIMS_12M_MEDIUM,Deux sinistres client sur 12 mois,37367,37367,448404,12.000000
6,Regles metier - Chronologie,DECLARATION_BEFORE_CLAIM_DATE,Declaration avant date de sinistre,51109,51109,408553,7.993758
20,Regles metier - Recurrence vehicule,VEHICLE_CLAIMS_12M_HIGH,Recurrence vehicule elevee sur 12 mois,26573,26573,398595,15.000000
21,Regles metier - Recurrence vehicule,VEHICLE_CLAIMS_12M_MEDIUM,Deux sinistres vehicule sur 12 mois,36201,36201,362010,10.000000
8,Regles metier - Chronologie,LONG_DECLARATION_DELAY_MEDIUM,Delai de declaration a examiner,56658,56658,283265,4.999559
10,Regles metier - Montant atypique,AMOUNT_LOW_BY_GUARANTEE,Montant a surveiller dans la garantie,32805,32805,196830,6.000000


## 8. Verification des plafonds et ponderations

Les points business rules sont configures par poids et plafonnes par famille. Ce controle verifie que les sorties ne depassent pas les plafonds declares dans le JSON.

In [10]:
rule_details = details[details["signal_family"].astype(str).str.startswith("Regles metier -")].copy()
rule_details["rule_family"] = rule_details["signal_family"].str.replace("Regles metier - ", "", regex=False)
family_points_by_claim = rule_details.groupby(["claim_sk", "rule_family"])["points"].sum().reset_index()
family_points_by_claim = family_points_by_claim.merge(family_caps, on="rule_family", how="left")
family_cap_checks = family_points_by_claim.assign(over_cap=lambda df: df["points"] > df["family_cap_points"])

business_total_by_claim = rule_details.groupby("claim_sk")["points"].sum().reset_index(name="business_rule_points")
business_max = int(score_config["business_rules"]["max_points"])

cap_validation = pd.DataFrame([{
    "family_cap_violation_rows": int(family_cap_checks["over_cap"].sum()),
    "business_rule_total_over_cap_rows": int((business_total_by_claim["business_rule_points"] > business_max).sum()),
    "business_rule_max_configured": business_max,
    "business_rule_max_observed": int(business_total_by_claim["business_rule_points"].max()) if not business_total_by_claim.empty else 0,
}])

assert cap_validation.loc[0, "family_cap_violation_rows"] == 0
assert cap_validation.loc[0, "business_rule_total_over_cap_rows"] == 0

display(cap_validation)
display(family_cap_checks.sort_values("points", ascending=False).head(20))

,family_cap_violation_rows,business_rule_total_over_cap_rows,business_rule_max_configured,business_rule_max_observed
0,0,0,70,70


,claim_sk,rule_family,points,family_cap_points,over_cap
315571,242756,Recurrence client,25,25,False
443931,328183,Recurrence client,25,25,False
443935,328184,Recurrence client,25,25,False
306813,236500,Recurrence client,25,25,False
45,33,Recurrence client,25,25,False
348265,265855,Recurrence client,25,25,False
528109,373458,Recurrence client,25,25,False
21867,12005,Recurrence client,25,25,False
429383,319699,Recurrence client,25,25,False
543534,381891,Recurrence client,25,25,False


## 9. Audit post-inspection Scenario A

Le score hybride doit utiliser uniquement le Scenario A valide. Scenario B reste hors points et hors details de score.

In [11]:
post_details = details[details["signal_family"].eq("Post-inspection")].copy()
post_summary = (
    post_details.groupby(["signal_code", "signal_label"], dropna=False)
    .agg(rows=("claim_sk", "size"), distinct_claims=("claim_sk", "nunique"), total_points=("points", "sum"), max_points=("points", "max"))
    .reset_index()
    if not post_details.empty else pd.DataFrame(columns=["signal_code", "signal_label", "rows", "distinct_claims", "total_points", "max_points"])
)

post_source = read_sql("""
    SELECT scenario_code, confidence_level, delay_bucket, defective_zone,
           COUNT(*) AS rows,
           COUNT(DISTINCT claim_sk) AS distinct_claims
    FROM mart.fact_post_inspection_attention_signal
    WHERE signal_version = :version
      AND signal_run_id = :run_id
    GROUP BY scenario_code, confidence_level, delay_bucket, defective_zone
    ORDER BY rows DESC
""", {"version": POST_INSPECTION_SIGNAL_VERSION, "run_id": post_inspection_run_id})

display(post_summary)
display(post_source.head(30))

,signal_code,signal_label,rows,distinct_claims,total_points,max_points
0,POST_INSPECTION_CONTEXT_ONLY,Contexte technique post-inspection documente,11,11,0,0
1,POST_INSPECTION_HIGH,Verification post-inspection prioritaire suggeree,24,24,600,25
2,POST_INSPECTION_MEDIUM,Signal post-inspection a examiner,25,25,375,15


,scenario_code,confidence_level,delay_bucket,defective_zone,rows,distinct_claims
0,A_INSPECTION_TO_CLAIM,MEDIUM,DAYS_31_90,SOUS_VEHICULE,21,21
1,A_INSPECTION_TO_CLAIM,MEDIUM,DAYS_31_90,TOUR_DU_VEHICULE,17,17
2,A_INSPECTION_TO_CLAIM,MEDIUM,DAYS_31_90,INTERIEUR,14,14
3,A_INSPECTION_TO_CLAIM,HIGH,DAYS_8_30,TOUR_DU_VEHICULE,12,12
4,A_INSPECTION_TO_CLAIM,HIGH,DAYS_8_30,SOUS_CAPOT,11,11
5,A_INSPECTION_TO_CLAIM,MEDIUM,DAYS_31_90,ENTRETIEN,10,10
6,A_INSPECTION_TO_CLAIM,HIGH,DAYS_8_30,SOUS_VEHICULE,9,8
7,A_INSPECTION_TO_CLAIM,HIGH,DAYS_8_30,INTERIEUR,9,9
8,A_INSPECTION_TO_CLAIM,LOW,DAYS_31_90,NO_DOCUMENTED_ANOMALY,8,8
9,A_INSPECTION_TO_CLAIM,HIGH,DAYS_8_30,ENTRETIEN,7,7


## 10. Comparaison avec Claim Attention V1

Cette comparaison sert a expliquer l'apport du score hybride par rapport a la baseline V1. Le V1 n'est pas modifie ; on lit seulement son dernier run si disponible.

In [12]:
v1_run_id = latest_run("mart.fact_claim_attention_score", "score_run_id", "score_version", "IRIS_CLAIM_ATTENTION_V1_CANDIDATE")
if v1_run_id:
    v1_scores = read_sql("""
        SELECT claim_sk, attention_score AS v1_attention_score, attention_level AS v1_attention_level
        FROM mart.fact_claim_attention_score
        WHERE score_version = 'IRIS_CLAIM_ATTENTION_V1_CANDIDATE'
          AND score_run_id = :run_id
    """, {"run_id": v1_run_id})
    comparison = scores[["claim_sk", "attention_score", "attention_level"]].merge(v1_scores, on="claim_sk", how="inner")
    comparison["score_delta_hybrid_minus_v1"] = comparison["attention_score"] - comparison["v1_attention_score"]
    display(comparison[["attention_score", "v1_attention_score", "score_delta_hybrid_minus_v1"]].describe())
    display(comparison.groupby(["v1_attention_level", "attention_level"], dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False))
else:
    print("No V1 score run found.")

,attention_score,v1_attention_score,score_delta_hybrid_minus_v1
count,381893.000000,381893.000000,381893.000000
mean,15.518496,10.775243,4.743253
std,15.503271,10.373021,7.649288
min,0.000000,0.000000,0.000000
25%,5.000000,0.000000,0.000000
50%,10.000000,8.000000,0.000000
75%,24.000000,18.000000,10.000000
max,92.000000,65.000000,52.000000


,v1_attention_level,attention_level,rows
0,Analyse standard,Analyse standard,287129
6,Points a verifier,Points a verifier,40113
2,Analyse standard,Points a verifier,38394
5,Points a verifier,Examen renforce suggere,13983
1,Analyse standard,Examen renforce suggere,1397
3,Examen renforce suggere,Examen renforce suggere,875
4,Points a verifier,Examen prioritaire suggere,2


## 11. Exemples de dossiers explicables

Les exemples ci-dessous sont destines a la revue avec l'encadrant ou le metier. Chaque dossier montre le score, les raisons principales et les signaux detail associes.

In [13]:
def explain_claim(claim_sk: int) -> pd.DataFrame:
    claim_score = scores[scores["claim_sk"].eq(claim_sk)].copy()
    claim_details = details[details["claim_sk"].eq(claim_sk)].copy().sort_values(["points", "signal_code"], ascending=[False, True])
    display(claim_score.T)
    return claim_details[["signal_family", "signal_code", "signal_label", "signal_value", "points", "severity", "business_explanation"]]

example_claims = scores.sort_values("attention_score", ascending=False).head(5)["claim_sk"].tolist()
post_claims = post_details.sort_values("points", ascending=False)["claim_sk"].drop_duplicates().head(5).tolist()
standard_claims = scores[scores["attention_score"].eq(0)].head(5)["claim_sk"].tolist()

print("Top score examples:", example_claims)
print("Post-inspection examples:", post_claims)
print("Standard examples:", standard_claims)

Top score examples: [5057, 337709, 351910, 243150, 249]
Post-inspection examples: [1795, 5057, 6863, 7631, 12717]
Standard examples: [468, 472, 475, 478, 482]


In [14]:
# Exemple 1: dossier avec score eleve
explain_claim(int(example_claims[0]))

,4593
claim_sk,5057
claim_business_id,G25511000006366|REM
score_version,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE
score_run_id,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE_20260...
feature_run_id,IRIS_CLAIM_ATTENTION_FEATURES_V1_CANDIDATE_202...
attention_score,92
attention_level,Examen prioritaire suggere
confidence_level,HIGH
main_reason_1,Verification post-inspection prioritaire suggeree
main_reason_2,Montant eleve dans la garantie


,signal_family,signal_code,signal_label,signal_value,points,severity,business_explanation
646722,Post-inspection,POST_INSPECTION_HIGH,Verification post-inspection prioritaire suggeree,"{""min_delay_days"": 20, ""signal_count"": 1, ""str...",25,HIGH,Un signal post-inspection Scenario A est dispo...
9902,Regles metier - Montant atypique,AMOUNT_HIGH_BY_GUARANTEE,Montant eleve dans la garantie,"{'ratio': 3.1012, 'percentile': 0.81162, 'high...",20,HIGH,Le montant evalue se situe nettement au-dessus...
9903,Regles metier - Recurrence client,CLIENT_CLAIMS_12M_HIGH,Recurrence client elevee sur 12 mois,3,20,HIGH,Plusieurs sinistres client precedents sont obs...
9904,Regles metier - Recurrence vehicule,VEHICLE_CLAIMS_12M_HIGH,Recurrence vehicule elevee sur 12 mois,3,15,HIGH,Plusieurs sinistres precedents sont observes s...
9905,Regles metier - Recurrence conducteur,DRIVER_CLAIMS_12M_HIGH,Recurrence conducteur sur 12 mois,4121,10,MEDIUM,Plusieurs sinistres precedents sont rattaches ...
9906,Regles metier - Recurrence conducteur,DRIVER_RECENT_PREVIOUS_CLAIM,Sinistre conducteur precedent recent,1,2,LOW,Le dossier suit de pres un sinistre precedent ...


In [15]:
# Exemple 2: dossier avec signal post-inspection, si disponible
if post_claims:
    display(explain_claim(int(post_claims[0])))
else:
    print("No post-inspection detail in this hybrid score run.")

,1330
claim_sk,1795
claim_business_id,G25511000002234|REM
score_version,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE
score_run_id,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE_20260...
feature_run_id,IRIS_CLAIM_ATTENTION_FEATURES_V1_CANDIDATE_202...
attention_score,35
attention_level,Points a verifier
confidence_level,HIGH
main_reason_1,Verification post-inspection prioritaire suggeree
main_reason_2,Recurrence conducteur sur 12 mois


,signal_family,signal_code,signal_label,signal_value,points,severity,business_explanation
646721,Post-inspection,POST_INSPECTION_HIGH,Verification post-inspection prioritaire suggeree,"{""min_delay_days"": 29, ""signal_count"": 3, ""str...",25,HIGH,Un signal post-inspection Scenario A est dispo...
2298,Regles metier - Recurrence conducteur,DRIVER_CLAIMS_12M_HIGH,Recurrence conducteur sur 12 mois,3,10,MEDIUM,Plusieurs sinistres precedents sont rattaches ...


In [16]:
# Exemple 3: dossier standard, score 0 si disponible
if standard_claims:
    display(explain_claim(int(standard_claims[0])))
else:
    print("No zero-score claim found.")

,3
claim_sk,468
claim_business_id,G25511000000576|REM
score_version,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE
score_run_id,IRIS_CLAIM_ATTENTION_HYBRID_V1_CANDIDATE_20260...
feature_run_id,IRIS_CLAIM_ATTENTION_FEATURES_V1_CANDIDATE_202...
attention_score,0
attention_level,Analyse standard
confidence_level,HIGH
main_reason_1,Aucun signal prioritaire hybride
main_reason_2,NaN


,signal_family,signal_code,signal_label,signal_value,points,severity,business_explanation


## 12. Export local des rapports notebook

Ces exports sont locaux et ne modifient pas PostgreSQL. Ils servent a conserver une trace de l'audit execute dans le notebook.

In [17]:
if EXPORT_REPORTS:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    validation_checks.to_csv(REPORT_DIR / "hybrid_notebook_validation_checks.csv", index=False)
    score_distribution.to_csv(REPORT_DIR / "hybrid_notebook_score_distribution.csv", index=False)
    confidence_distribution.to_csv(REPORT_DIR / "hybrid_notebook_confidence_distribution.csv", index=False)
    family_summary.to_csv(REPORT_DIR / "hybrid_notebook_signal_family_summary.csv", index=False)
    rule_summary.to_csv(REPORT_DIR / "hybrid_notebook_rule_summary.csv", index=False)
    cap_validation.to_csv(REPORT_DIR / "hybrid_notebook_cap_validation.csv", index=False)
    post_summary.to_csv(REPORT_DIR / "hybrid_notebook_post_inspection_summary.csv", index=False)
    run_context.to_csv(REPORT_DIR / "hybrid_notebook_run_context.csv", index=False)

REPORT_DIR

WindowsPath('c:/Users/wiem/Downloads/Projet PFE/IRIS_AUTO_FRAUD/data/quality_reports/scoring/claim_attention_hybrid_v1/notebook_audit')

## 13. Decision de validation

A renseigner apres execution du notebook :

```text
Decision technique : GO / PARTIAL / BLOCKED
Decision metier : GO pour demonstration / PARTIAL pour calibration / BLOCKED
```

Decision recommandee si tous les checks passent :

```text
GO technique pour Claim Attention Hybrid V1 Candidate.
PARTIAL metier jusqu'a validation des ponderations avec BNA Assurances.
```

Justification :

- score complet au grain dossier ;
- regles configurables ;
- score regle pondere ;
- post-inspection Scenario A integre ;
- Scenario B exclu des points ;
- details explicatifs disponibles ;
- aucune conclusion automatique.